# Pipeline: Outreach → Donation Impact

## Problem framing
**Who cares**: Founder / donor growth lead.

**Business question**: Which social media post choices (platform, type, topic, CTA, time) are most associated with **donation referrals** and **donation value**, so the team can post with purpose.

**Modeling goals**:
- **Predictive**: predict `donation_referrals` (count) and/or `estimated_donation_value_php`.
- **Explanatory**: identify the strongest drivers to inform content strategy.

## Primary targets
- `donation_referrals` (count; we’ll model as regression and also as a binary “high referral” classifier)
- `estimated_donation_value_php` (regression)

## Outputs
- `predictions_outreach_posts.csv`
- Driver tables (feature importance / coefficients)

> Note: This dataset contains post-level simulated donation attribution fields; in production you’d validate with real tracking (UTMs + donation funnel analytics).

## Data & feature notes

- **Unit of analysis**: one row per social media post.
- **Targets**:
  - `estimated_donation_value_php`: simulated attributed donation value (regression)
  - `donation_referrals`: simulated count of attributed donations (count-like)
- **Features**: post metadata (platform/type/topic), timing, CTA flags, and engagement metrics.

## Leakage / interpretation caveat
Features like `click_throughs`, `reach`, and `engagement_rate` often occur **after** publishing; they are great for explaining *what happened* and for optimizing distribution/creative, but they can inflate performance if you claim they’re known “before posting”.

If you want a strict *pre-post* planning model, drop post-performance features and use only `platform`, `post_hour`, `post_type`, `content_topic`, `has_call_to_action`, etc.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplcache"))

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "social_media_posts.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "social_media_posts.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate social_media_posts.csv from current working directory")

posts = pd.read_csv(DATA_DIR / "social_media_posts.csv")
donations = pd.read_csv(DATA_DIR / "donations.csv")

donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce")

# Optional attribution consistency check: donations.referral_post_id links back to posts
posts.shape, posts.columns[:10]

((812, 39),
 Index(['post_id', 'platform', 'platform_post_id', 'post_url', 'created_at',
        'day_of_week', 'post_hour', 'post_type', 'media_type', 'caption'],
       dtype='object'))

In [2]:
# Attribution check (optional): compare post table donation fields vs donation rows with referral_post_id

if "referral_post_id" in donations.columns:
    d_sm = donations.dropna(subset=["referral_post_id"]).copy()
    d_sm["referral_post_id"] = pd.to_numeric(d_sm["referral_post_id"], errors="coerce")
    d_sm["estimated_value"] = pd.to_numeric(d_sm["estimated_value"], errors="coerce")

    agg = d_sm.groupby("referral_post_id").agg(
        donations_n=("donation_id", "count"),
        est_value_sum=("estimated_value", "sum"),
    ).reset_index().rename(columns={"referral_post_id": "post_id"})

    chk = posts[["post_id", "donation_referrals", "estimated_donation_value_php"]].merge(agg, on="post_id", how="left")
    display(chk.head(10))
    print("Correlation: post.donation_referrals vs aggregated donations_n")
    print(pd.to_numeric(chk["donation_referrals"], errors="coerce").corr(pd.to_numeric(chk["donations_n"], errors="coerce")))
else:
    print("donations.referral_post_id not found; skipping attribution check")

,post_id,donation_referrals,estimated_donation_value_php,donations_n,est_value_sum
0,318,10,21473.25,NaN,NaN
1,529,2,4708.45,NaN,NaN
2,86,0,0.00,NaN,NaN
3,380,0,0.00,NaN,NaN
4,425,2,8351.49,NaN,NaN
5,807,1,3516.65,NaN,NaN
6,757,6,27029.10,NaN,NaN
7,554,14,55256.28,NaN,NaN
8,736,2,6056.45,NaN,NaN
9,148,1,4623.84,NaN,NaN


Correlation: post.donation_referrals vs aggregated donations_n
0.31594253267347155


In [3]:
# Basic EDA / sanity checks
cols = [
    "platform",
    "day_of_week",
    "post_hour",
    "post_type",
    "media_type",
    "has_call_to_action",
    "call_to_action_type",
    "content_topic",
    "sentiment_tone",
    "is_boosted",
    "boost_budget_php",
    "caption_length",
    "num_hashtags",
    "mentions_count",
    "impressions",
    "reach",
    "likes",
    "comments",
    "shares",
    "saves",
    "click_throughs",
    "engagement_rate",
    "donation_referrals",
    "estimated_donation_value_php",
]
cols = [c for c in cols if c in posts.columns]

display(posts[cols].describe(include="all").T.head(25))
print("Missing % (top):")
print((posts[cols].isna().mean() * 100).sort_values(ascending=False).head(10).round(2))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
platform,812,7,Facebook,199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,812,7,Tuesday,136,NaN,NaN,NaN,NaN,NaN,NaN,NaN
post_hour,812.0,NaN,NaN,NaN,12.690887,6.296557,0.0,8.0,13.0,18.0,23.0
post_type,812,6,ImpactStory,203,NaN,NaN,NaN,NaN,NaN,NaN,NaN
media_type,812,5,Photo,227,NaN,NaN,NaN,NaN,NaN,NaN,NaN
has_call_to_action,812,2,True,493,NaN,NaN,NaN,NaN,NaN,NaN,NaN
call_to_action_type,493,4,LearnMore,131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
content_topic,812,9,Education,126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sentiment_tone,812,6,Informative,162,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_boosted,812,2,False,685,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Missing % (top):
boost_budget_php       84.36
call_to_action_type    39.29
platform                0.00
mentions_count          0.00
donation_referrals      0.00
engagement_rate         0.00
click_throughs          0.00
saves                   0.00
shares                  0.00
comments                0.00
dtype: float64


In [4]:
# Prepare modeling table

df = posts.copy()

# Keep rows with targets present
TARGET_REG = "estimated_donation_value_php"
TARGET_CNT = "donation_referrals"

for t in [TARGET_REG, TARGET_CNT]:
    df[t] = pd.to_numeric(df[t], errors="coerce")

# A simple binary target: top 30% of donation_referrals
q = df[TARGET_CNT].quantile(0.70)
df["high_referrals"] = (df[TARGET_CNT] >= q).astype(int)

feature_cols = [
    "platform",
    "day_of_week",
    "post_hour",
    "post_type",
    "media_type",
    "has_call_to_action",
    "call_to_action_type",
    "content_topic",
    "sentiment_tone",
    "is_boosted",
    "boost_budget_php",
    "caption_length",
    "num_hashtags",
    "mentions_count",
    "impressions",
    "reach",
    "likes",
    "comments",
    "shares",
    "saves",
    "click_throughs",
    "engagement_rate",
    "follower_count_at_post",
]
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()

y_value = df[TARGET_REG].copy()
y_referrals = df[TARGET_CNT].copy()

# Ensure categorical columns are plain objects (avoids pandas string dtype edge-cases in sklearn imputers)
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
for c in cat_cols:
    X[c] = X[c].astype("object")

display(X.head())
print("Targets:", y_value.notna().mean(), y_referrals.notna().mean())

,platform,day_of_week,post_hour,post_type,media_type,has_call_to_action,call_to_action_type,content_topic,sentiment_tone,is_boosted,boost_budget_php,caption_length,num_hashtags,mentions_count,impressions,reach,likes,comments,shares,saves,click_throughs,engagement_rate,follower_count_at_post
0,WhatsApp,Thursday,18,FundraisingAppeal,Text,True,LearnMore,Education,Grateful,False,NaN,157,0,3,1580,1093,118,36,22,9,48,0.1105,1522
1,Instagram,Friday,11,EducationalContent,Photo,False,NaN,Education,Celebratory,False,NaN,150,4,0,6362,4395,548,110,149,59,85,0.1745,1833
2,LinkedIn,Sunday,10,EventPromotion,Text,False,NaN,Reintegration,Urgent,False,NaN,138,0,0,554,336,27,7,12,4,3,0.1411,457
3,Instagram,Monday,15,ThankYou,Video,False,NaN,Education,Emotional,False,NaN,113,2,1,4309,2948,190,55,45,16,35,0.0677,1796
4,TikTok,Monday,15,ThankYou,Reel,True,LearnMore,Education,Hopeful,True,4030.64,129,4,1,23175,14008,728,232,141,79,474,0.0802,916


Targets: 1.0 1.0


In [5]:
# Train a baseline regression model for donation value

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in numeric_cols]

pre = ColumnTransformer(
    [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

mask = y_value.notna()
X_tr, X_te, y_tr, y_te = train_test_split(X.loc[mask], y_value.loc[mask], test_size=0.2, random_state=42)

# Compare models
models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
}

best_name, best_score, best_pipe = None, -1e9, None
for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    cv = cross_val_score(p, X_tr, y_tr, cv=5, scoring="r2")
    print(f"{name}: CV R²={cv.mean():.3f} ± {cv.std():.3f}")
    if cv.mean() > best_score:
        best_name, best_score, best_pipe = name, cv.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_tr, y_tr)
pred = best_pipe.predict(X_te)

pipe = best_pipe

print("Test MAE:", mean_absolute_error(y_te, pred))
print("Test R²:", r2_score(y_te, pred))

Ridge: CV R²=0.137 ± 0.531


RandomForest: CV R²=0.517 ± 0.160

Best model: RandomForest
Test MAE: 26463.693581565592
Test R²: 0.4835728065673818


In [6]:
# Feature importance (handles both Ridge coefficients and RF importances)

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if len(cat_cols) else []
feature_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    vals = model.coef_
    label = "coef"
else:
    vals = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feature_names, label: vals}).sort_values(label, key=abs, ascending=False)
print(f"Top 20 features by |{label}| ({best_name})")
display(imp_df.head(20))

Top 20 features by |importance| (RandomForest)


,feature,importance
9,shares,0.509870
19,platform_WhatsApp,0.064426
32,post_type_ImpactStory,0.046990
8,comments,0.042691
7,likes,0.039910
2,caption_length,0.028865
0,post_hour,0.028213
11,click_throughs,0.023264
4,mentions_count,0.022999
12,engagement_rate,0.022577


In [7]:
# Save post-level predictions

pred_value_all = pipe.predict(X)

out = df[["post_id", "platform", "post_type", "content_topic", "created_at"]].copy()
out["pred_estimated_donation_value_php"] = pred_value_all
out["donation_referrals"] = y_referrals
out["estimated_donation_value_php"] = y_value

out_path = DATA_DIR / "predictions_outreach_posts.csv"
out.to_csv(out_path, index=False)
out_path

PosixPath('/Users/masonzarges/Desktop/INTEX Data/predictions_outreach_posts.csv')

## Decisions you can make from this

- Use the **top positive categorical coefficients** (platform/post_type/topic/CTA) to design a content calendar.
- Use **click_throughs** and **CTA presence** as operational levers to test quickly.
- Validate by re-running monthly and checking stability by platform (avoid overfitting to one channel).

In [8]:
print("Rows:", len(df))
print("Date range:", posts["created_at"].min(), "→", posts["created_at"].max())
print("Saved:", out_path)

# Show the highest predicted value posts (useful for content planning)
display(out.sort_values("pred_estimated_donation_value_php", ascending=False).head(10))

Rows: 812
Date range: 2023-01-05 18:52:00 → 2026-02-26 21:56:00
Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_outreach_posts.csv


,post_id,platform,post_type,content_topic,created_at,pred_estimated_donation_value_php,donation_referrals,estimated_donation_value_php
495,266,WhatsApp,ImpactStory,SafehouseLife,2024-11-26 10:05:00,1.528696e+06,458,2402435.96
795,338,Facebook,ImpactStory,SafehouseLife,2026-02-07 10:25:00,9.553629e+05,162,283414.58
524,765,TikTok,ImpactStory,AwarenessRaising,2025-01-17 19:28:00,9.517742e+05,239,1148927.49
457,211,Facebook,ImpactStory,Health,2024-10-09 11:03:00,8.113518e+05,203,808353.61
23,292,YouTube,ImpactStory,Reintegration,2023-02-14 16:41:00,6.906135e+05,197,801434.24
43,159,TikTok,FundraisingAppeal,SafehouseLife,2023-03-12 19:08:00,4.597375e+05,63,294622.66
676,502,TikTok,ImpactStory,Health,2025-09-05 23:44:00,4.131220e+05,109,311938.91
480,652,LinkedIn,ImpactStory,DonorImpact,2024-11-13 10:07:00,3.957217e+05,85,555828.12
627,180,Instagram,ImpactStory,Health,2025-06-09 21:59:00,3.904953e+05,101,214655.68
496,449,YouTube,FundraisingAppeal,AwarenessRaising,2024-11-28 17:50:00,3.759385e+05,206,236730.39
